# 36. Histogram efficiency/background maps from a ROOT file

**Objectives:**
- Write small synthetic ROOT `TH2` histograms with `uproot` (this package only
  *reads* ROOT histograms, see `docs/root_io.md`).
- Read them back with `read_root_histogram2d`.
- Build `HistogramEfficiency`/`HistogramBackground` maps from the file with
  `histogram_efficiency_from_root`/`histogram_background_from_root`.

Run cells top to bottom in a fresh kernel. Masses are in GeV, invariants in GeV²,
daughter indices start at zero. `dalitzplotfitter` has no TH2 writer of its own
(production efficiency/background maps normally come from an external
ROOT-based tool); to exercise the reading API without depending on another
notebook's output file, this notebook writes its own tiny `TH2` histograms
with plain `uproot`, using the same `f["name"] = (values, x_edges, y_edges)`
API already exercised in [tutorial 37](tutorial_37_root_low_level_io.ipynb).

In [1]:
from dalitzplotfitter import enable_x64
enable_x64()  # Must precede numerical work: amplitudes use complex128.

from pathlib import Path

import numpy as np
import uproot
from dalitzplotfitter import (
    DecayChannel, DecayModel, NonResonant, RealImag, Resonance,
    histogram_background_from_root, histogram_efficiency_from_root,
    read_root_histogram2d,
)

output_dir = Path(".")

## 1. Write a synthetic efficiency and background `TH2` with `uproot`

Both histograms live in ordinary Dalitz coordinates, `(s12, s13)` in GeV²,
for a `B+ -> K+ pi+ pi-` model with `parent_mass = 5.279` GeV. The efficiency
histogram values are clipped to `[0, 1]`; the background histogram is any
non-negative shape.

In [2]:
channel = DecayChannel("B+", ("K+", "pi+", "pi-"))
m_max = channel.parent_mass ** 2

s12_edges = np.linspace(0.0, m_max, 7)
s13_edges = np.linspace(0.0, m_max, 6)

rng = np.random.default_rng(36)
efficiency_values = rng.uniform(0.5, 0.95, size=(len(s12_edges) - 1, len(s13_edges) - 1))
background_values = rng.uniform(0.5, 2.0, size=(len(s12_edges) - 1, len(s13_edges) - 1))

hist_path = output_dir / "tutorial_36_histograms.root"
with uproot.recreate(hist_path) as root_file:
    root_file["efficiency_map"] = (efficiency_values, s12_edges, s13_edges)
    root_file["background_map"] = (background_values, s12_edges, s13_edges)
print("Wrote", hist_path, "size:", hist_path.stat().st_size, "bytes")

Wrote tutorial_36_histograms.root size: 13915 bytes


## 2. `read_root_histogram2d`: the low-level reader

Returns `(values, x_edges, y_edges)` with over/underflow excluded — this is
the building block every `*_from_root` helper below calls internally.

In [3]:
values, x_edges, y_edges = read_root_histogram2d(hist_path, "efficiency_map")
np.testing.assert_allclose(np.asarray(values), efficiency_values)
np.testing.assert_allclose(np.asarray(x_edges), s12_edges)
np.testing.assert_allclose(np.asarray(y_edges), s13_edges)
print("Round trip confirmed:", values.shape, "bins")

Round trip confirmed: (6, 5) bins


## 3. `histogram_efficiency_from_root` and `histogram_background_from_root`

These wrap `read_root_histogram2d` and construct plain-Dalitz
`HistogramEfficiency`/`HistogramBackground` objects directly, given which
invariant each TH2 axis represents (`x_variable`/`y_variable`).

In [4]:
efficiency = histogram_efficiency_from_root(
    hist_path, "efficiency_map", x_variable="s12", y_variable="s13",
)
background = histogram_background_from_root(
    hist_path, "background_map", x_variable="s12", y_variable="s13",
)

model = DecayModel(
    channel,
    [
        Resonance("Kstar", (0, 2), RealImag(1.0, 0.0), mass=0.892, width=0.051, spin=1),
        NonResonant(RealImag(0.4, -0.2), name="NR"),
    ],
    normalization_method="square-dalitz", normalization_pair=(0, 2),
    normalization_resolution=40,
)
grid = model.normalization_sample.as_dict()
efficiency_at_grid = np.asarray(efficiency(grid))
background_at_grid = np.asarray(background(grid))
print(f"efficiency(grid): min={efficiency_at_grid.min():.3f}, max={efficiency_at_grid.max():.3f}")
print(f"background(grid): min={background_at_grid.min():.3f}, max={background_at_grid.max():.3f}")
assert np.all((efficiency_at_grid >= 0.0) & (efficiency_at_grid <= 1.0))
assert np.all(background_at_grid >= 0.0)

efficiency(grid): min=0.564, max=0.920
background(grid): min=0.557, max=1.892


## Try it yourself

1. Try `folded=True` with `x_edges == y_edges` (required for folding) and
   compare `efficiency(grid)` before and after.
2. Build `SquareDalitzHistogramEfficiency` from a ROOT file with
   `square_dalitz_efficiency_from_root` instead, on a histogram whose axes
   are `(m', theta')`.
3. Pass `efficiency=efficiency` straight into a `FitSession`, as in
   [tutorial 35](tutorial_35_histogram_maps_from_arrays.ipynb).

## Continue learning

Reference: [ROOT I/O](../../docs/root_io.md), [backgrounds and vetoes](../../docs/backgrounds_and_vetoes.md).

Return to [the course guide](TUTORIALS.md).